# Mat & Comp – ML 8: Regresión logística
## ¿Qué tan seguro estás?

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 7

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar por qué una recta no sirve para predecir un «sí o no» y cómo la **función sigmoide** lo arregla.
2. Interpretar la salida de la regresión logística como una **probabilidad** y encontrar su **frontera de decisión**.
3. Medir el error de un clasificador con la **entropía cruzada** y entender por qué castiga tanto al que se equivoca muy seguro.
4. Entrenar una regresión logística **a mano** con descenso de gradiente (ML 3) y luego con scikit-learn.
5. Mover el **umbral** de decisión para cambiar el equilibrio entre precisión y sensibilidad (ML 7), y clasificar más de dos clases.

## 🔍 «70 % de probabilidad de lluvia»

El pronóstico del tiempo no dice «mañana llueve» o «mañana no llueve». Dice «hay un **70 %** de probabilidad de lluvia». Con eso decides si llevas paraguas.

En ML 4 y ML 5, k vecinos y los árboles respondían «aprobó» o «no aprobó», sin más.

✏️ *Un estudiante estudia 6 horas a la semana y otro estudia 12. Un modelo dice «aprobará» para ambos. ¿Estás igual de seguro en los dos casos? ¿Qué información extra te gustaría que diera el modelo?*

**Mi respuesta:**

Hoy construiremos un modelo que responde con una **probabilidad**: la **regresión logística**. A pesar de su nombre, es un modelo para **clasificar**. Además, es la neurona más famosa del machine learning: la volverás a encontrar en la Sección 4.

## 1. Los datos: ¿aprobó?

Volvemos a los 60 estudiantes de ML 1. Como en ML 4, decimos que un estudiante **aprobó** si su nota es al menos 3.0. Esta vez escribimos la respuesta como un número: **1 = aprobó**, **0 = no aprobó**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(URL)
x = datos["horas_estudio"].to_numpy()
y = (datos["nota_matematicas"] >= 3.0).astype(int).to_numpy()   # 1 = aprobó

print(len(x), "estudiantes;", y.sum(), "aprobaron y", len(y) - y.sum(), "no")

plt.figure(figsize=(7, 3.2))
plt.scatter(x, y, c=y, cmap="coolwarm_r", edgecolor="k", alpha=0.7)
plt.yticks([0, 1], ["0: no aprobó", "1: aprobó"])
plt.xlabel("horas de estudio por semana")
plt.title("¿Aprobó matemáticas?")
plt.grid(alpha=0.3)
plt.show()

✏️ *Mirando la gráfica: ¿a partir de cuántas horas casi todos aprueban? ¿Hay una zona donde se mezclan los dos grupos?*

## 2. ¿Y si usamos una recta?

En ML 2 ajustamos rectas con mínimos cuadrados. Intentemos lo mismo con los ceros y unos:

In [ ]:
m, c = np.polyfit(x, y, 1)
xs = np.linspace(-2, 18, 200)

plt.figure(figsize=(7, 3.2))
plt.scatter(x, y, c=y, cmap="coolwarm_r", edgecolor="k", alpha=0.7)
plt.plot(xs, m * xs + c, color="purple", label="recta de mínimos cuadrados")
plt.axhspan(1, 1.6, color="gray", alpha=0.15)
plt.axhspan(-0.6, 0, color="gray", alpha=0.15)
plt.xlabel("horas de estudio por semana")
plt.ylim(-0.6, 1.6)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for h in [0, 6, 16]:
    print(f"{h:2} horas: la recta predice {m * h + c:.2f}")

Problemas:
- Con 16 horas la recta predice más de 1, y con 0 horas predice menos de 0. Una **probabilidad** no puede ser 1.2 ni −0.1.
- La recta sigue subiendo para siempre, aunque después de cierto punto ya todos aprueban.

Necesitamos una curva que vaya de 0 a 1, en forma de **S**.

## 3. La función sigmoide

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Recibe cualquier número $z$ y devuelve un número entre 0 y 1.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3))
plt.plot(z, sigmoide(z))
plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.axvline(0, color="gray", ls="--", lw=1)
plt.xlabel("z"); plt.ylabel("σ(z)")
plt.title("La sigmoide")
plt.grid(alpha=0.3)
plt.show()

for valor in [-6, -2, -1, 0, 1, 2, 6]:
    print(f"σ({valor:2}) = {sigmoide(valor):.4f}")

Observa:
- $\sigma(0) = 0.5$: en $z = 0$ el modelo está en duda total.
- Si $z$ es muy positivo, $\sigma(z)$ se acerca a 1; si es muy negativo, a 0. Nunca llega del todo.
- Es simétrica: $\sigma(-z) = 1 - \sigma(z)$. Compara $\sigma(2)$ y $\sigma(-2)$.

### El modelo

La regresión logística toma la recta de siempre, $z = w x + b$, y la pasa por la sigmoide:

$$p = \sigma(w x + b) = \text{probabilidad de que } y = 1$$

Así, $w$ decide qué tan **empinada** es la S (y hacia qué lado sube), y $b$ la **corre** a izquierda o derecha.

In [ ]:
xs = np.linspace(0, 14, 200)
fig, (e1, e2) = plt.subplots(1, 2, figsize=(11, 3.2))
for w in [0.3, 1, 3]:
    e1.plot(xs, sigmoide(w * (xs - 6)), label=f"w = {w}")
e1.set_title("Cambiando w (centro en 6 horas)")
for centro in [3, 6, 9]:
    e2.plot(xs, sigmoide(1 * (xs - centro)), label=f"centro en {centro} h")
e2.set_title("Cambiando b (w = 1)")
for e in (e1, e2):
    e.set_xlabel("horas"); e.set_ylabel("p"); e.legend(); e.grid(alpha=0.3)
plt.show()

(Escribimos $w(x - 6)$, que es lo mismo que $wx + b$ con $b = -6w$.)

### La frontera de decisión

Para decidir «sí» o «no», el modelo responde «aprobó» si $p \ge 0.5$. Como $\sigma(z) = 0.5$ justo cuando $z = 0$, la **frontera** está donde

$$w x + b = 0 \quad\Longrightarrow\quad x = -\frac{b}{w}$$

## 4. ¿Cómo medir el error? La entropía cruzada

Si un estudiante **aprobó** ($y = 1$) y el modelo le dio probabilidad $p$:
- con $p = 0.9$ el modelo estaba bien y seguro;
- con $p = 0.4$ estaba algo equivocado;
- con $p = 0.01$ estaba **muy seguro y muy equivocado**.

La **entropía cruzada** mide esto con logaritmos:

$$\text{error} = \begin{cases} -\ln p & \text{si } y = 1 \\ -\ln(1 - p) & \text{si } y = 0 \end{cases}$$

En una sola fórmula: $-\big[y \ln p + (1 - y) \ln(1 - p)\big]$. El error del modelo es el promedio sobre todos los estudiantes.

In [ ]:
def entropia_cruzada(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print("El estudiante aprobó (y = 1):")
for p in [0.99, 0.9, 0.6, 0.4, 0.1, 0.01]:
    ec = entropia_cruzada(np.array([1]), np.array([p]))
    print(f"  p = {p:4}:  entropía cruzada = {ec:5.2f}   error cuadrático = {(1 - p) ** 2:.2f}")

In [ ]:
p = np.linspace(0.005, 0.995, 200)
plt.figure(figsize=(6, 3.2))
plt.plot(p, -np.log(p), label="entropía cruzada: −ln p")
plt.plot(p, (1 - p) ** 2, label="error cuadrático: (1 − p)²")
plt.xlabel("probabilidad que el modelo dio a «aprobó»")
plt.ylabel("error")
plt.title("Error cuando el estudiante sí aprobó")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

El error cuadrático nunca pasa de 1, pero la entropía cruzada **se dispara** cuando el modelo se equivoca estando muy seguro. Así el modelo aprende a no ser arrogante. 🙃

## 5. Entrenar a mano con descenso de gradiente

En ML 3 bajamos la «montaña» del error moviendo los parámetros contra la pendiente. Para la entropía cruzada, las pendientes salen sorprendentemente sencillas (se demuestran con cálculo; aquí las usaremos):

$$\frac{\partial E}{\partial w} = \text{promedio de } (p - y)\,x \qquad \frac{\partial E}{\partial b} = \text{promedio de } (p - y)$$

$p - y$ es «cuánto se pasó» el modelo con cada estudiante.

In [ ]:
w, b = 0.0, 0.0
alfa = 0.1
historial = []

for paso in range(30001):
    p = sigmoide(w * x + b)
    E = entropia_cruzada(y, p)
    historial.append(E)
    w -= alfa * np.mean((p - y) * x)
    b -= alfa * np.mean(p - y)
    if paso % 5000 == 0:
        print(f"paso {paso:5}: E = {E:.4f}   w = {w:.3f}   b = {b:.3f}")

frontera = -b / w
print(f"\nFrontera de decisión: {frontera:.2f} horas")

In [ ]:
fig, (e1, e2) = plt.subplots(1, 2, figsize=(12, 3.5))
e1.plot(historial)
e1.set_xscale("log")
e1.set_xlabel("paso (escala logarítmica)"); e1.set_ylabel("entropía cruzada")
e1.set_title("Curva de aprendizaje")
e1.grid(alpha=0.3)

xs = np.linspace(0, 14, 200)
e2.scatter(x, y, c=y, cmap="coolwarm_r", edgecolor="k", alpha=0.7)
e2.plot(xs, sigmoide(w * xs + b), color="purple", lw=2, label="p = σ(wx + b)")
e2.axvline(frontera, color="gray", ls="--", label=f"frontera: {frontera:.1f} h")
e2.set_xlabel("horas de estudio"); e2.set_ylabel("probabilidad de aprobar")
e2.legend(); e2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for h in [3, 6, 9, 12]:
    print(f"{h:2} horas: probabilidad de aprobar = {sigmoide(w * h + b):.1%}")

¡Ahora el modelo sí responde la pregunta inicial! Con 6 horas está en duda (cerca de la frontera); con 12 horas está casi seguro.

✏️ *¿Cuántos estudiantes clasifica bien el modelo? Calcula `np.mean((sigmoide(w * x + b) >= 0.5) == y)`.*

## 6. Con scikit-learn

`LogisticRegression` hace todo esto por nosotros. Dos detalles:
- Como en ML 4, `X` debe ser una **tabla** (una columna por característica), por eso el `reshape(-1, 1)`.
- scikit-learn, por defecto, frena un poco los pesos para evitar el sobreajuste (se llama **regularización** y la veremos en ML 9). Con `C=1e6` casi no frena, así que debe dar lo mismo que nuestro descenso de gradiente.

In [ ]:
from sklearn.linear_model import LogisticRegression

X = x.reshape(-1, 1)
modelo = LogisticRegression(C=1e6)
modelo.fit(X, y)

print(f"scikit-learn: w = {modelo.coef_[0, 0]:.3f}   b = {modelo.intercept_[0]:.3f}")
print(f"A mano:       w = {w:.3f}   b = {b:.3f}")
print(f"Exactitud (con los mismos datos): {modelo.score(X, y):.0%}")

Hay dos formas de preguntarle al modelo:
- `predict` da la **clase** (0 o 1);
- `predict_proba` da las **probabilidades** de cada clase: una columna para 0 y otra para 1.

In [ ]:
nuevos = np.array([[3], [6], [9], [12]])
print("Clases:", modelo.predict(nuevos))
print("Probabilidades [no aprobó, aprobó]:")
print(modelo.predict_proba(nuevos).round(3))

## 7. Dos características: una frontera recta

Ahora usemos horas de estudio **y** horas de sueño, como en ML 4. El modelo es

$$p = \sigma(w_1 x_1 + w_2 x_2 + b)$$

y la frontera $w_1 x_1 + w_2 x_2 + b = 0$ es una **recta** en el plano. Como en ML 4, quitamos los estudiantes con datos faltantes y normalizamos con un `Pipeline`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d2 = datos.dropna(subset=["horas_sueno"])
X2 = d2[["horas_estudio", "horas_sueno"]].to_numpy()
y2 = (d2["nota_matematicas"] >= 3.0).astype(int).to_numpy()

modelo2 = make_pipeline(StandardScaler(), LogisticRegression())
modelo2.fit(X2, y2)

# mapa de probabilidad
g1, g2 = np.meshgrid(np.linspace(0, 14, 200), np.linspace(4.5, 10, 200))
P = modelo2.predict_proba(np.c_[g1.ravel(), g2.ravel()])[:, 1].reshape(g1.shape)

plt.figure(figsize=(7, 5))
mapa = plt.contourf(g1, g2, P, levels=20, cmap="coolwarm_r", alpha=0.6)
plt.colorbar(mapa, label="probabilidad de aprobar")
plt.contour(g1, g2, P, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm_r", edgecolor="k")
plt.xlabel("horas de estudio"); plt.ylabel("horas de sueño")
plt.title("Regresión logística: la frontera p = 0.5 es una recta")
plt.show()

Compara con el mapa de k vecinos de ML 4: aquel tenía bordes irregulares; este es una franja que pasa suavemente del rojo al azul. Comparemos su exactitud con **validación cruzada** (ML 7):

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

candidatos = {
    "regresión logística": make_pipeline(StandardScaler(), LogisticRegression()),
    "k vecinos (k = 5)": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "árbol (profundidad 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
}
for nombre, m in candidatos.items():
    puntajes = cross_val_score(m, X2, y2, cv=5)
    print(f"{nombre:22}: {puntajes.mean():.1%}")

Con tan pocos datos las diferencias son pequeñas y cambian si cambian los grupos de la validación cruzada. Pero la regresión logística tiene dos ventajas: da **probabilidades** y es fácil de **interpretar**.

In [ ]:
w1, w2 = modelo2[-1].coef_[0]
print(f"Peso de las horas de estudio: {w1:.2f}")
print(f"Peso de las horas de sueño:   {w2:.2f}")

Como normalizamos, los pesos se pueden comparar: el estudio pesa más del doble que el sueño en este modelo. Ambos son positivos: más horas, más probabilidad de aprobar.

## 8. El umbral: ¿desde qué probabilidad decimos «sí»?

No siempre conviene usar 0.5. Imagina que el colegio quiere ofrecer **tutorías** a quien probablemente **no** apruebe. Es peor dejar a alguien sin ayuda que ofrecérsela a alguien que no la necesitaba. Entonces conviene marcar como «en riesgo» a más estudiantes.

Recuerda de ML 7: la **sensibilidad** mide cuántos de los que no aprobaron detectamos, y la **precisión** cuántos de los marcados realmente no aprobaron.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

# Probabilidad de NO aprobar, con validación cruzada para ser honestos (ML 7)
p_aprobar = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()),
                              X2, y2, cv=5, method="predict_proba")[:, 1]
riesgo_real = (y2 == 0)

print("umbral   marcados  detectados/reales   sensibilidad  precisión")
for umbral in [0.3, 0.5, 0.7, 0.9]:
    marcado = p_aprobar < umbral          # en riesgo si la probabilidad de aprobar es baja
    vp = np.sum(marcado & riesgo_real)
    sens = vp / riesgo_real.sum()
    prec = vp / marcado.sum()
    print(f"  {umbral}      {marcado.sum():2}          {vp:2}/{riesgo_real.sum()}             {sens:5.0%}        {prec:5.0%}")

Al subir el umbral marcamos a más estudiantes: detectamos a casi todos los que lo necesitan (sube la sensibilidad), pero también ofrecemos tutoría a algunos que no la necesitaban (baja la precisión). **Elegir el umbral es una decisión humana**, que depende de qué error es más costoso.

✏️ *Si las tutorías fueran muy caras y solo hubiera cupo para 10 estudiantes, ¿qué umbral elegirías?*

## 9. Más de dos clases: los dígitos

¿Y si hay 10 clases, como los dígitos de ML 4 y ML 6? La regresión logística calcula un puntaje $z_k$ para **cada** clase (cada una con sus propios pesos) y los convierte en probabilidades que suman 1 con la función **softmax**:

$$p_k = \frac{e^{z_k}}{e^{z_0} + e^{z_1} + \cdots + e^{z_9}}$$

scikit-learn lo hace automáticamente cuando hay más de dos clases.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digitos = load_digits()
X_ent, X_pru, y_ent, y_pru = train_test_split(digitos.data / 16, digitos.target,
                                              test_size=0.25, random_state=0)
modelo_d = LogisticRegression(max_iter=1000)
modelo_d.fit(X_ent, y_ent)
print(f"Exactitud en prueba: {modelo_d.score(X_pru, y_pru):.1%}")

probs = modelo_d.predict_proba(X_pru[:1])[0]
print("Probabilidades para la primera imagen de prueba:", probs.round(3))
print("Suman:", probs.sum().round(6), "  Respuesta:", probs.argmax(), "  Real:", y_pru[0])

Cada clase tiene 64 pesos, uno por píxel. Si los dibujamos como imágenes de 8 × 8, vemos **qué busca** el modelo para cada dígito: en azul, los píxeles que suman a favor; en rojo, los que restan.

In [ ]:
fig, ejes = plt.subplots(1, 10, figsize=(15, 2))
lim = np.abs(modelo_d.coef_).max()
for k, eje in enumerate(ejes):
    eje.imshow(modelo_d.coef_[k].reshape(8, 8), cmap="RdBu", vmin=-lim, vmax=lim)
    eje.set_title(k)
    eje.axis("off")
plt.suptitle("Pesos de cada clase (azul: a favor, rojo: en contra)")
plt.show()

Las imágenes son borrosas, pero tienen pistas. ✏️ *En el 0, ¿de qué color es el centro? ¿Por qué tiene sentido, si un cero tiene un hueco en el medio? En el 1, ¿dónde están los píxeles azules?*

## 10. Errores comunes

**Error 1: pasar un solo número en lugar de una tabla.** scikit-learn siempre espera una tabla (2D), aunque sea de una sola fila y una sola columna:

In [ ]:
modelo.predict_proba(6)

La solución: `modelo.predict_proba([[6]])` (una fila con una columna).

**Error 2: confundir las columnas de `predict_proba`.** La columna 0 es la probabilidad de la clase 0 (no aprobó), y la 1 la de la clase 1. Si tienes dudas, mira `modelo.classes_`.

**Error 3: usar regresión lineal para clasificar.** Como vimos en la sección 2, da «probabilidades» menores que 0 o mayores que 1.

**Error 4: creer que una probabilidad alta es una garantía.** «90 % de aprobar» significa que de cada 10 estudiantes así, más o menos 1 **no** aprueba. El modelo sabe lo que muestran los datos, nada más.

**Error 5: llamar «logística» a la sigmoide y «regresión» al modelo.** No es un error, pero confunde: la regresión logística **clasifica**. El nombre es histórico.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …calculas `sigmoide(50)` y `sigmoide(-50)`? ¿Y `entropia_cruzada(np.array([1]), sigmoide(np.array([-50.0])))`?
2. …usas `alfa = 1.0` en el descenso de gradiente de la sección 5? ¿Y `alfa = 0.001`?
3. …entrenas con `LogisticRegression(C=0.01)` en la sección 6? ¿Qué le pasa a `w`? ¿Y a las probabilidades?
4. …cambias `>= 3.0` por `>= 4.0` (aprobar con 4)? ¿Dónde queda la frontera?
5. …en la sección 7 usas solo `horas_sueno` como característica? ¿Cuál es la exactitud con validación cruzada?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). La sigmoide con calculadora.** Calcula $\sigma(0)$, $\sigma(2)$ y $\sigma(-2)$ con calculadora (recuerda $e \approx 2.718$). Comprueba con Python que $\sigma(-2) = 1 - \sigma(2)$.

**Ejercicio 2 (fácil). Leer el modelo.** Con el `modelo` de la sección 6: ¿cuál es la probabilidad de aprobar con 4 horas? ¿Y con 8? ¿Cuántas horas hacen falta para tener al menos 90 % de probabilidad? (Pista: busca $x$ tal que $wx + b = \ln 9$, porque $\sigma(\ln 9) = 0.9$.)

**Ejercicio 3 (medio). Las redes sociales.** Entrena una regresión logística con `horas_redes` como única característica. ¿El peso $w$ es positivo o negativo? ¿Qué significa? ¿Cuál es la probabilidad de aprobar con 0 horas de redes y con 6 horas?

**Ejercicio 4 (medio). El mejor umbral.** Con las probabilidades `p_aprobar` de la sección 8, calcula la exactitud para los umbrales 0.1, 0.2, …, 0.9 y grafica exactitud contra umbral. ¿Cuál umbral da la mayor exactitud? ¿Es siempre 0.5?

**Ejercicio 5 (avanzado). Dos variables a mano.** Escribe el descenso de gradiente para el modelo de la sección 7 con numpy: normaliza `X2` (resta el promedio y divide por la desviación estándar de cada columna), usa un vector de pesos `w` de tamaño 2, calcula `p = sigmoide(Xn @ w + b)` y las pendientes `Xn.T @ (p - y2) / n` y `np.mean(p - y2)`. Compara tus pesos con los de `LogisticRegression(C=1e6)` sobre los mismos datos normalizados.

## 🏆 Reto opcional: cuando la frontera recta no alcanza

`make_moons(200, noise=0.2, random_state=0)` (de `sklearn.datasets`) crea dos medias lunas entrelazadas.

1. Entrena una regresión logística y dibuja su mapa de probabilidad como en la sección 7. ¿Qué exactitud logra con validación cruzada?
2. Agrega **nuevas características** calculadas: $x_1^2$, $x_2^2$, $x_1 x_2$, $x_1^3$, $x_2^3$. (Pista: `PolynomialFeatures(degree=3)` de `sklearn.preprocessing`, dentro de un `make_pipeline`.) ¿Cómo cambia la frontera? ¿Y la exactitud?

## 🎬 Cierre

Hoy aprendiste que:
- Para predecir un «sí o no» con una probabilidad, la recta $wx + b$ se pasa por la **sigmoide**, que la convierte en un número entre 0 y 1.
- La **frontera de decisión** está donde $wx + b = 0$; con dos características es una recta.
- La **entropía cruzada** castiga mucho al modelo que se equivoca estando muy seguro.
- Se entrena con **descenso de gradiente**, con pendientes muy simples: promedio de $(p - y)x$.
- El **umbral** se elige según qué error es más costoso.
- Con la **softmax**, la regresión logística clasifica más de dos clases, y sus pesos se pueden dibujar.

**Para pensar:** en el reto, inventar características nuevas como $x^2$ permitió fronteras curvas. ¿Y si queremos predecir la **nota** (un número) usando varias características a la vez y también potencias de ellas? Eso es la **regresión múltiple y polinomial** de la próxima lección, donde también veremos cómo evitar que tantas características lleven al sobreajuste: la **regularización**. 📐